# Día 10 · Unidad 11 — Ejemplo aplicado: riesgo de suscripción de vida (Prudential, Kaggle)

**Objetivos de aprendizaje**
- Ver un segundo caso completo de ML aplicado a seguros, esta vez de clasificación multiclase (8 niveles de riesgo).
- Practicar limpieza de nulos guiada por umbrales, análisis de correlación con la variable objetivo y EDA con boxplots/heatmaps.
- Ajustar una regresión logística con `Pipeline` (`StandardScaler` + `LogisticRegression`) y leer su matriz de confusión.

**Duración estimada:** 60-75 minutos (más orientado a lectura y ejecución guiada que a escribir código nuevo).

**Requisitos previos:** `02_teoria_pipelines_riesgos_ml.ipynb` de este mismo día.

---

> **Origen del dataset**: este notebook usa el dataset público de la competición de Kaggle [*Prudential Life Insurance Assessment*](https://www.kaggle.com/c/prudential-life-insurance-assessment/overview) (Prudential Financial, ~2015). **No es un dato real de ninguna aseguradora del curso ni de ningún asegurado real del Colegio de Actuarios de Cataluña** — es un dataset de terceros, público, incluido aquí únicamente como segundo ejemplo aplicado de clasificación en el dominio de seguros, con una estructura de variables distinta a la del resto del curso (más de cien variables anonimizadas `Product_Info_*`, `Medical_History_*`, etc., sin descripción de negocio detallada por el propio concurso).

## 1. El problema

Prudential Life Insurance ofreció esta competición para automatizar parte de la evaluación de riesgo en la suscripción de seguros de vida. El dataset original contiene más de cien variables por solicitante (edad, altura/peso/IMC, historial médico codificado, información de la póliza solicitada...) y una variable objetivo, `Response`, que clasifica el riesgo en **8 niveles ordinales** (1 = mayor riesgo, 8 = menor riesgo).

Trabajamos con un subconjunto ya reducido (`data/shortset/train.csv`, incluido en este mismo directorio) para que el notebook se ejecute en segundos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()

train = pd.read_csv("data/shortset/train.csv")
train = train.set_index("Id")
train.index.name = None
train.head()

## 2. Limpieza de datos

### 2.1 Nulos

Con más de cien variables, no tiene sentido revisarlas una a una: descartamos primero las que tienen demasiados nulos para ser útiles (más de un 30% de valores ausentes), y el resto las imputamos.

In [ ]:
def get_cols_with_nulls(df: pd.DataFrame) -> pd.Series:
    """Devuelve la proporcion de nulos de cada columna que tenga al menos uno.

    Args:
        df: DataFrame a inspeccionar.

    Returns:
        pd.Series: proporcion de nulos (0-1) por columna, ordenada de mayor a menor.
    """
    null_share = df.isna().sum(axis=0).sort_values(ascending=False)
    null_share = null_share[null_share > 0] / df.shape[0]
    return null_share


def drop_columns_with_many_nulls(df: pd.DataFrame, threshold: float) -> pd.DataFrame:
    """Elimina las columnas cuya proporcion de nulos supera `threshold`.

    Args:
        df: DataFrame de entrada.
        threshold: proporcion de nulos (0-1) a partir de la cual se descarta la columna.

    Returns:
        pd.DataFrame: copia de `df` sin esas columnas.
    """
    null_share = get_cols_with_nulls(df)
    cols_to_drop = null_share[null_share > threshold].index
    return df.drop(columns=cols_to_drop)


train = drop_columns_with_many_nulls(train, threshold=0.3)

null_stats = pd.concat(
    [
        train[get_cols_with_nulls(train).index].describe().T.round(2),
        get_cols_with_nulls(train).to_frame(name="proporcion_nulos"),
    ],
    axis=1,
)
null_stats

Antes de rellenar los nulos restantes, veamos la distribución de una de las variables con más nulos:

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(data=train, x="Family_Hist_4", kde=True)
plt.title("Distribucion de Family_Hist_4 (antes de imputar)")
plt.show()

In [ ]:
# Imputacion simple con la media, solo en columnas numericas (float64)
train = train.apply(lambda col: col.fillna(col.mean()) if col.dtype == "float64" else col, axis=0)
print("Nulos restantes:", train.isna().sum().sum())

> ⚠️ **Error típico**: rellenar nulos con la media (o la mediana) de **todo** el dataset, incluyendo lo que después será el conjunto de test, es exactamente el mismo problema de fuga de datos del notebook anterior — aquí se hace así por simplicidad didáctica (es fiel al ejemplo original), pero en un proyecto real esta imputación debería ir dentro de un `Pipeline`/`ColumnTransformer` con un `SimpleImputer`, ajustado solo con el train, como en `02_teoria_pipelines_riesgos_ml.ipynb`.

### 2.2 Correlación con la variable objetivo

Con decenas de variables numéricas, priorizamos: nos quedamos solo con las que tienen una correlación relevante (en valor absoluto) con `Response`.

In [ ]:
train_corr = train.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(train_corr, vmin=-1, vmax=1, center=0, cmap="RdBu", ax=ax)
ax.set_title("Correlacion entre todas las variables numericas")
plt.show()

In [ ]:
def filter_columns_by_correlation(df: pd.DataFrame, corr_with_target: pd.Series, threshold: float) -> pd.DataFrame:
    """Filtra columnas de `df` cuya correlacion con el target supera `threshold` en valor absoluto.

    Args:
        df: DataFrame con todas las columnas candidatas.
        corr_with_target: correlacion de cada columna con la variable objetivo.
        threshold: umbral de correlacion absoluta a partir del cual se conserva la columna.

    Returns:
        pd.DataFrame: subconjunto de columnas de `df` que superan el umbral.
    """
    corr_cols = corr_with_target[(corr_with_target >= threshold) | (corr_with_target <= -threshold)].index
    return df[corr_cols]


corr_con_target = train_corr["Response"]
X = filter_columns_by_correlation(train, corr_con_target, threshold=0.2)
print("Columnas conservadas:", list(X.columns))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(X.corr(), vmin=-1, vmax=1, center=0, cmap="RdBu", annot=True, fmt=".2f", ax=ax)
plt.title("Correlacion entre las variables filtradas")
plt.show()

## 3. EDA de las variables filtradas

### `Product_Info_4` frente a `Response`

In [ ]:
def boxplot_con_puntos(df: pd.DataFrame, x: str, y: str, ax) -> None:
    """Dibuja un boxplot con los puntos individuales superpuestos (semitransparentes).

    Args:
        df: DataFrame con los datos.
        x: columna categorica/ordinal para el eje X.
        y: columna numerica para el eje Y.
        ax: Axes de matplotlib donde dibujar.
    """
    sns.boxplot(ax=ax, data=df, x=x, y=y, boxprops={"alpha": 0.3}, whis=np.inf)
    sns.stripplot(ax=ax, data=df, x=x, y=y, alpha=0.1)


fig, ax = plt.subplots(figsize=(9, 5))
boxplot_con_puntos(X, x="Response", y="Product_Info_4", ax=ax)
plt.show()

Los niveles de riesgo más bajos (`Response` alto = menor riesgo) tienden a valores más bajos de `Product_Info_4` — probablemente una variable relacionada con el precio o las condiciones de la póliza solicitada (el dataset no documenta el significado exacto de cada variable, solo su categoría).

Repetimos el mismo gráfico para el resto de variables numéricas filtradas:

In [ ]:
columnas_numericas_filtradas = X.select_dtypes("float64").columns
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(13, 10))
for ax, columna in zip(axes.ravel(), columnas_numericas_filtradas):
    boxplot_con_puntos(X, x="Response", y=columna, ax=ax)
plt.tight_layout()
plt.show()

### Variables categóricas frente a `Response`

Las columnas de tipo `int64` que quedan son en realidad variables binarias (0/1) — el dataset ya viene "dummyficado" de origen por Kaggle.

In [ ]:
X = X.copy()
columnas_categoricas = [col for col in X.columns if "Medical_" in col]
X[columnas_categoricas] = X[columnas_categoricas].astype("category")
X["Response"] = X["Response"].astype("category")

X.dtypes

In [ ]:
def heatmap_cruzado(df: pd.DataFrame, x: str, y: str, ax) -> None:
    """Dibuja un heatmap de frecuencias relativas cruzadas entre dos columnas categoricas.

    Args:
        df: DataFrame con los datos.
        x: primera columna categorica.
        y: segunda columna categorica.
        ax: Axes de matplotlib donde dibujar.
    """
    cross_table = pd.crosstab(df[x], df[y])
    cross_table = (cross_table / cross_table.sum().sum()).round(4)
    sns.heatmap(cross_table, annot=True, cmap="Blues", fmt="g", ax=ax)


fig, axes = plt.subplots(nrows=1, ncols=len(columnas_categoricas), figsize=(5 * len(columnas_categoricas), 4.5))
if len(columnas_categoricas) == 1:
    axes = [axes]
for ax, columna in zip(axes, columnas_categoricas):
    heatmap_cruzado(X, x="Response", y=columna, ax=ax)
plt.tight_layout()
plt.show()

Como último paso de limpieza, descartamos algunas categorías atípicas de las variables médicas que podrían introducir ruido en la estimación (siguiendo el criterio del análisis original):

In [ ]:
for columna in columnas_categoricas:
    valores_raros = X[columna].value_counts()
    valores_raros = valores_raros[valores_raros < 5].index
    if len(valores_raros) > 0:
        X = X[~X[columna].isin(valores_raros)].copy()

print("Filas tras descartar categorias con muy pocas observaciones:", X.shape[0])

## 4. Modelo: regresión logística multiclase

`Response` tiene 8 niveles ordinales, así que este es un problema de **clasificación multiclase** (no binaria): `LogisticRegression` de scikit-learn lo gestiona automáticamente sin cambiar de clase, ajustando internamente una estrategia multinomial.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X.drop(columns="Response"), X["Response"], test_size=0.30, random_state=101, stratify=X["Response"]
)

pipeline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
pipeline.fit(X_train, y_train)
print("Pipeline entrenado.")

`make_pipeline()` es un atajo de `Pipeline(steps=[...])` que asigna nombres automáticamente a partir de la clase de cada paso (`"standardscaler"`, `"logisticregression"`) — útil cuando no necesitas nombres personalizados para `RandomizedSearchCV` como en el notebook anterior.

### Coeficientes del modelo

In [ ]:
coeficientes = pd.DataFrame({
    "variable": X_train.columns,
    "coeficiente": pipeline["logisticregression"].coef_[0],
}).sort_values("coeficiente", ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=coeficientes, y="variable", x="coeficiente", hue="variable", legend=False, palette="coolwarm", ax=ax)
plt.title("Coeficientes de la primera clase del modelo multiclase")
plt.show()

> ⚠️ **Error típico**: en un modelo multiclase, `.coef_` tiene una fila **por cada clase** (aquí, hasta 8), no una sola. El gráfico anterior muestra solo `coef_[0]` (los coeficientes asociados a la primera clase frente al resto) a modo ilustrativo — leer solo esa fila y sacar conclusiones sobre "el modelo" en general, sin mirar el resto de clases, es un error de interpretación frecuente.

### Evaluación del modelo

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score

predicciones = pipeline.predict(X_test)

matriz_confusion = pd.crosstab(
    pd.Series(predicciones, name="Prediccion"),
    pd.Series(y_test.values, name="Real"),
)
print("Matriz de confusion - valores absolutos")
display(matriz_confusion)

print("\nMatriz de confusion - valores relativos")
display((matriz_confusion / matriz_confusion.sum().sum()).round(3))

accuracy = accuracy_score(y_test, predicciones)
print(f"\nAccuracy en test: {accuracy * 100:.1f}%")

La mayor parte de los aciertos y errores se concentran cerca de la diagonal — el modelo suele equivocarse en niveles de riesgo **adyacentes** (confundir el nivel 7 con el 8) más que en extremos opuestos, lo que tiene sentido porque `Response` es ordinal: una regresión logística "a secas" (sin explotar esa estructura de orden) ya recoge parte de esa continuidad de forma implícita, aunque no esté diseñada específicamente para variables ordinales.

**Para ti:** con `pd.crosstab(...)`, calcula qué porcentaje de las predicciones erróneas se equivocan en como mucho un nivel de diferencia (p. ej. predecir 7 cuando el real es 8) frente a las que se equivocan en más de un nivel. ¿Cambia tu valoración de "cómo de malo" es el modelo?

**Para ti:** repite el `train_test_split` con un `random_state` distinto (por ejemplo `7`) y vuelve a entrenar. ¿Cuánto varía el accuracy? Si varía bastante con un cambio tan pequeño, ¿qué te dice eso sobre el tamaño de este subconjunto de datos?

## Resumen

- Con datasets de muchas variables, filtrar primero por proporción de nulos y luego por correlación con el objetivo es una forma razonable de reducir dimensionalidad antes de modelar — aunque, como se ha señalado, imputar o filtrar con el dataset completo (antes de separar train/test) es una simplificación didáctica que en un proyecto real debería resolverse dentro de un `Pipeline`.
- `LogisticRegression` gestiona automáticamente problemas multiclase (aquí, 8 niveles de riesgo) sin cambiar de API respecto al caso binario.
- En un modelo multiclase, `.coef_` tiene una fila por clase — hay que tener cuidado de no interpretar una sola fila como "el" comportamiento del modelo.
- La matriz de confusión de un problema ordinal (como niveles de riesgo) se lee mejor fijándose en si los errores son "cercanos" en la escala o "lejanos", no solo en el porcentaje de aciertos exacto.
- Este dataset es un caso público de Kaggle: útil para practicar el flujo completo de un problema de clasificación en seguros, pero sin significado de negocio verificable variable a variable (a diferencia del dataset de anulación de pólizas del notebook anterior, que sí es del propio dominio documentado del curso).

**Siguiente:** `04_teoria_rentas.ipynb`, primer bloque de la Unidad 12 (actuarial-financiero cuantitativo).